# OGGM-VAS with RGI 7.0: CMIP-forced reconstruction and projections

**What this notebook does**
1. Bias-corrects each CMIP experiment (GCM x scenario) against CRU and stores it per glacier.
2. `HISTORICAL_FORCING = "CMIP"`: runs VAS 1901 -> 2015 with the CMIP *historical* forcing (ends Dec 2014), then continues 2015 -> 2100 with the scenario forcing (starts Jan 2015).
   `HISTORICAL_FORCING = "CRU"`: one shared CRU reconstruction 1901 -> 2019, then CMIP projections from 2019.
3. Aggregates to regional volume / area / length / sea-level equivalent, plots them in official IPCC colours, saves CSVs.

**Where to put your CMIP data:** only in the cell called **"CMIP DATA - EDIT THIS CELL"** (cell 3).
Every entry accepts a direct link, a local path, a wildcard, or a list of files.

Everything is re-runnable: finished steps are detected on disk and skipped.

In [ ]:
# ======================================================================
# 1. IMPORTS AND OGGM 1.6.3 / VAS COMPATIBILITY PATCHES
# ======================================================================
import hashlib
import inspect
import json
import os
import re
import time
from dataclasses import dataclass, fields
from pathlib import Path
from typing import Optional, Sequence, Tuple, Union
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import requests
import matplotlib.pyplot as plt

import oggm.cfg as cfg
from oggm import utils, workflow, tasks
from oggm import GlacierDirectory
from oggm.shop import gcm_climate

# --- old VAS climate_info basename -----------------------------------------
if "climate_info" not in cfg.BASENAMES:
    cfg.BASENAMES["climate_info"] = (
        "climate_info.json",
        "VAS glacier climate and calibration information.",
    )


# --- JSON compatibility patch for OGGM 1.6.3 + VAS --------------------------
def write_json_compat(self, var, filename, filesuffix=""):
    def np_convert(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

    with open(self.get_filepath(filename, filesuffix=filesuffix), "w") as f:
        json.dump(var, f, default=np_convert)


GlacierDirectory.write_json = write_json_compat

# --- get_climate_info compatibility patch (merge old VAS climate_info.json) -
_original_get_climate_info = GlacierDirectory.get_climate_info


def get_climate_info_compat(self, input_filesuffix=""):
    out = _original_get_climate_info(self, input_filesuffix=input_filesuffix)
    try:
        vas_info = self.read_json("climate_info")
        if isinstance(vas_info, dict):
            out.update(vas_info)
    except FileNotFoundError:
        pass
    return out


GlacierDirectory.get_climate_info = get_climate_info_compat

import oggm_vas as vascaling


In [ ]:
# ======================================================================
# 2. CONFIGURATION
# ======================================================================
# ------------------------------------------------------------
# Region (RGI 7.0): 1-19
# ------------------------------------------------------------
RGI_VERSION = "70G"
RGI_REGION = 15

RGI_REGION_NAMES = {
    1: "Alaska", 2: "Western Canada and USA", 3: "Arctic Canada North",
    4: "Arctic Canada South", 5: "Greenland Periphery", 6: "Iceland",
    7: "Svalbard", 8: "Scandinavia", 9: "Russian Arctic", 10: "North Asia",
    11: "Central Europe", 12: "Caucasus and Middle East", 13: "Central Asia",
    14: "South Asia West", 15: "South Asia East", 16: "Low Latitudes",
    17: "Southern Andes", 18: "New Zealand", 19: "Antarctic and Subantarctic",
}

# ------------------------------------------------------------
# Which forcing drives the historical run (1901 -> split year)?
#   "CMIP": each experiment is forced by its own CMIP historical data up to
#           Dec 2014, then continues with its scenario from Jan 2015.
#   "CRU" : one shared CRU reconstruction up to CRU_RECON_END_YEAR; the CMIP
#           scenarios start from that reconstruction.
# CRU is always used as the reference for the bias correction and the VAS calibration.
# ------------------------------------------------------------
HISTORICAL_FORCING = "CMIP"

MODEL_START_YEAR = 1901
CRU_RECON_END_YEAR = 2019           # only used when HISTORICAL_FORCING = "CRU"
FUTURE_END_YEAR = 2100
RUN_FUTURE = True

# CMIP calendar join (CMIP6: historical ends Dec 2014, scenarios start Jan 2015)
CMIP_HISTORICAL_END = (2014, 12)    # (year, month), inclusive
CMIP_SCENARIO_START = (2015, 1)     # (year, month), inclusive

# ------------------------------------------------------------
# GCM bias correction against CRU
# ------------------------------------------------------------
BIAS_CORRECTION_PERIOD = ("1961", "1990")
APPLY_BIAS_CORRECTION = True
SCALE_STDDEV = True

# ------------------------------------------------------------
# VAS calibration table (RGI6/CRU reference table shipped with OGGM-VAS)
# ------------------------------------------------------------
VAS_REFERENCE_TSTARS_FILE = "vas_ref_tstars_rgi6_cru4.csv"

# ------------------------------------------------------------
# Folders. CMIP files / downloads go to CMIP_DATA_DIR.
# ------------------------------------------------------------
WORKDIR = Path(os.path.expanduser(f"~/oggm_work/vas_rgi70_region{RGI_REGION:02d}"))
CMIP_DATA_DIR = Path(os.path.expanduser("~/CMIP_data"))

# ------------------------------------------------------------
# Run control
# ------------------------------------------------------------
MAX_GLACIERS = 20            # quick test. Set to None for the full region!
RECOMPUTE_BASE = False       # True: redo masks / CRU climate / calibration / CRU run
OVERWRITE_CMIP = False       # True: redo GCM bias correction even if gcm_data exists
REUSE_RESULTS = True         # True: reuse finished experiments (CSV files) instead of re-running

# ------------------------------------------------------------
# Physical constants
# ------------------------------------------------------------
RHO_ICE = 900.0
RHO_WATER = 1000.0
OCEAN_AREA_KM2 = 3.625e8

# ------------------------------------------------------------
# Derived settings and checks (do not edit)
# ------------------------------------------------------------
REGION_NAME = RGI_REGION_NAMES[RGI_REGION]
REGION_SLUG = REGION_NAME.replace(" ", "")
CMIP_MODE = HISTORICAL_FORCING.upper() == "CMIP"

if HISTORICAL_FORCING.upper() not in ("CMIP", "CRU"):
    raise ValueError('HISTORICAL_FORCING must be "CMIP" or "CRU".')
if RGI_REGION not in RGI_REGION_NAMES:
    raise ValueError("RGI_REGION must be between 1 and 19.")
if (CMIP_HISTORICAL_END[0] * 12 + CMIP_HISTORICAL_END[1]) + 1 != (
        CMIP_SCENARIO_START[0] * 12 + CMIP_SCENARIO_START[1]):
    raise ValueError("CMIP_SCENARIO_START must be the month right after CMIP_HISTORICAL_END.")

# year in which the historical run ends and the projection starts
# (state at the start of that year = after the last historical month)
SPLIT_YEAR = CMIP_SCENARIO_START[0] if CMIP_MODE else CRU_RECON_END_YEAR
if CMIP_MODE and CMIP_SCENARIO_START[1] != 1:
    raise ValueError("In CMIP mode the scenario must start in January.")
if FUTURE_END_YEAR <= SPLIT_YEAR or SPLIT_YEAR <= MODEL_START_YEAR:
    raise ValueError("Need MODEL_START_YEAR < split year < FUTURE_END_YEAR.")

CRU_HIST_SUFFIX = f"_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}"

WORKDIR.mkdir(parents=True, exist_ok=True)
CMIP_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Region        : RGI70-{RGI_REGION:02d} {REGION_NAME}")
print(f"Historical    : {HISTORICAL_FORCING.upper()} forcing, {MODEL_START_YEAR}-{SPLIT_YEAR}")
print(f"Projection    : {SPLIT_YEAR}-{FUTURE_END_YEAR}")
print(f"Work dir      : {WORKDIR}")
print(f"CMIP data dir : {CMIP_DATA_DIR}")
if MAX_GLACIERS is not None:
    print(f"TEST MODE     : only {MAX_GLACIERS} glaciers (set MAX_GLACIERS = None for the full run)")


In [ ]:
# ======================================================================
# 3. CMIP DATA - EDIT THIS CELL
# ======================================================================
# ##########################################################################
# #                                                                        #
# #   CMIP DATA - EDIT THIS CELL                                           #
# #   One entry per GCM x scenario. This is the only cell with CMIP links. #
# #                                                                        #
# ##########################################################################
#
# Every *_temp / *_prcp value can be:
#   - a direct link          "https://.../tas_Amon_..._historical_....nc"   (downloaded once, then cached)
#   - an OPeNDAP link        contains "/dodsC/"                              (read remotely, nothing saved)
#   - a local file           "C:/data/tas_ssp245.nc"  or  "~/CMIP_data/tas_ssp245.nc"
#   - a path relative to     "MPI-ESM1-2-HR/tas_ssp245.nc"  (looked up inside CMIP_DATA_DIR)
#   - a wildcard             "MPI-ESM1-2-HR/historical/tas_*.nc"  (CMIP6 files are often split in time)
#   - a list of the above    ["tas_1850-1949.nc", "tas_1950-2014.nc"]
#
# historical_* : CMIP historical run, must end in December 2014 (cut automatically)
# future_*     : scenario run, must start in January 2015 (cut automatically)
#                Leave historical_* out when the scenario file already contains the
#                history (e.g. the OGGM cmip5-ng files below).
#
# name     : unique, only letters / digits / . _ -   (used in file names)
# scenario : rcp26 rcp45 rcp60 rcp85 ssp119 ssp126 ssp245 ssp370 ssp434 ssp460 ssp534os ssp585
#            (decides the IPCC colour)
# model    : GCM name (decides the line style when you plot several GCMs)
# Optional keys: generation="CMIP6", label="...", color="#rrggbb", end_year=2099,
#                tas_var="tas", pr_var="pr", lat_name="lat", lon_name="lon", calendar="noleap"

CMIP5NG = "https://cluster.klima.uni-bremen.de/~oggm/cmip5-ng"


def cmip5ng(gcm, rcp):
    """CMIP5 from the OGGM server (links; historical + scenario in one file)."""
    return dict(
        name=f"{gcm}_{rcp}", model=gcm, scenario=rcp, generation="CMIP5",
        future_temp=f"{CMIP5NG}/tas/tas_mon_{gcm}_{rcp}_r1i1p1_g025.nc",
        future_prcp=f"{CMIP5NG}/pr/pr_mon_{gcm}_{rcp}_r1i1p1_g025.nc",
    )


CMIP_EXPERIMENTS = [

    # ---------------- CMIP5 from the OGGM server (works as it is) ----------------
    cmip5ng("CCSM4", "rcp26"),
    cmip5ng("CCSM4", "rcp45"),
    cmip5ng("CCSM4", "rcp85"),

    # ---------------- CMIP6 with direct links -----------------------------------
    # dict(
    #     name="MPI-ESM1-2-HR_ssp245", model="MPI-ESM1-2-HR", scenario="ssp245", generation="CMIP6",
    #     historical_temp="https://PASTE-LINK/tas_Amon_MPI-ESM1-2-HR_historical_r1i1p1f1_gn_185001-201412.nc",
    #     historical_prcp="https://PASTE-LINK/pr_Amon_MPI-ESM1-2-HR_historical_r1i1p1f1_gn_185001-201412.nc",
    #     future_temp="https://PASTE-LINK/tas_Amon_MPI-ESM1-2-HR_ssp245_r1i1p1f1_gn_201501-210012.nc",
    #     future_prcp="https://PASTE-LINK/pr_Amon_MPI-ESM1-2-HR_ssp245_r1i1p1f1_gn_201501-210012.nc",
    # ),

    # ---------------- CMIP6 with files on your computer --------------------------
    # dict(
    #     name="MPI-ESM1-2-HR_ssp585", model="MPI-ESM1-2-HR", scenario="ssp585", generation="CMIP6",
    #     historical_temp="~/CMIP_data/MPI-ESM1-2-HR/historical/tas_*.nc",
    #     historical_prcp="~/CMIP_data/MPI-ESM1-2-HR/historical/pr_*.nc",
    #     future_temp="~/CMIP_data/MPI-ESM1-2-HR/ssp585/tas_*.nc",
    #     future_prcp="~/CMIP_data/MPI-ESM1-2-HR/ssp585/pr_*.nc",
    # ),
]


In [ ]:
# ======================================================================
# 4. EXPERIMENT DEFINITION, IPCC COLOURS (no need to edit)
# ======================================================================
Source = Union[str, Path, Sequence[Union[str, Path]]]


@dataclass(frozen=True)
class CMIPExperiment:
    name: str
    scenario: str
    future_temp: Source
    future_prcp: Source
    historical_temp: Optional[Source] = None
    historical_prcp: Optional[Source] = None

    model: str = ""                  # GCM name
    generation: str = ""             # "CMIP5" / "CMIP6" (info only)
    source: str = ""
    label: str = ""                  # legend label (default from scenario)
    color: str = ""                  # plot colour (default: IPCC colour of the scenario)
    end_year: Optional[int] = None   # override FUTURE_END_YEAR for this GCM

    # join of historical and scenario files (only used when historical files are given)
    historical_end: Tuple[int, int] = CMIP_HISTORICAL_END
    scenario_start: Tuple[int, int] = CMIP_SCENARIO_START

    tas_var: str = "tas"
    pr_var: str = "pr"
    lat_name: str = "lat"
    lon_name: str = "lon"
    time_name: str = "time"

    tas_units: Optional[str] = None  # only if missing in the file
    pr_units: Optional[str] = None
    calendar: Optional[str] = None   # detected from the file when None
    time_unit: Optional[str] = None  # detected from the file when None

    bias_correction_period: Tuple[str, str] = BIAS_CORRECTION_PERIOD
    apply_bias_correction: bool = APPLY_BIAS_CORRECTION
    scale_stddev: bool = SCALE_STDDEV

    def __post_init__(self):
        if not re.fullmatch(r"[A-Za-z0-9._-]+", self.name):
            raise ValueError(
                f"Experiment name '{self.name}' may only contain letters, "
                "digits, '.', '_' and '-' (it is used in file names).")

    @property
    def gcm(self):
        if self.model:
            return self.model
        tail = f"_{self.scenario}"
        return self.name[: -len(tail)] if self.name.endswith(tail) else self.name

    @property
    def suffix(self):                # gcm_data file suffix
        return f"_{self.name}"

    @property
    def hist_suffix(self):           # CMIP-forced historical run
        return f"_hist_{self.name}"

    @property
    def future_suffix(self):         # projection run
        return f"_future_{self.name}"


def build_experiments(entries):
    allowed = {f.name for f in fields(CMIPExperiment)}
    experiments = []
    for entry in entries:
        unknown = set(entry) - allowed
        if unknown:
            raise KeyError(f"Unknown keys {sorted(unknown)} in experiment "
                           f"'{entry.get('name', '?')}'. Allowed: {sorted(allowed)}")
        for key, value in entry.items():
            if any(tok in str(value) for tok in ("PASTE-LINK", "<server>", "<link>")):
                raise ValueError(f"Experiment '{entry.get('name')}': '{key}' still contains "
                                 "a placeholder. Fill in the link or remove the entry.")
        experiments.append(CMIPExperiment(**entry))
    names = [e.name for e in experiments]
    if len(names) != len(set(names)):
        raise ValueError(f"Experiment names must be unique: {names}")
    return experiments


EXPERIMENTS = build_experiments(CMIP_EXPERIMENTS)
print("Experiments:", [e.name for e in EXPERIMENTS])

SCENARIO_LABELS = {
    "rcp26": "RCP2.6", "rcp45": "RCP4.5", "rcp60": "RCP6.0", "rcp85": "RCP8.5",
    "ssp119": "SSP1-1.9", "ssp126": "SSP1-2.6", "ssp245": "SSP2-4.5",
    "ssp370": "SSP3-7.0", "ssp434": "SSP4-3.4", "ssp460": "SSP4-6.0",
    "ssp534os": "SSP5-3.4OS", "ssp585": "SSP5-8.5",
}

# Official IPCC colours (same values as pyam.plotting.PYAM_COLORS, see
# https://pyam-iamc.readthedocs.io/en/stable/tutorials/ipcc_colors.html)
IPCC_COLORS = {
    "AR5-RCP-2.6": "#0000FF", "AR5-RCP-4.5": "#79BCFF",
    "AR5-RCP-6.0": "#FF822D", "AR5-RCP-8.5": "#FF0000",
    "AR6-RCP-2.6": "#003466", "AR6-RCP-4.5": "#709fcc",
    "AR6-RCP-6.0": "#c37900", "AR6-RCP-8.5": "#980002",
    "AR6-SSP1-1.9": "#00a9cf", "AR6-SSP1-2.6": "#003466",
    "AR6-SSP2-4.5": "#f69320", "AR6-SSP3-7.0": "#df0000",
    "AR6-SSP4-3.4": "#2274ae", "AR6-SSP4-6.0": "#b0724e",
    "AR6-SSP5-3.4-OS": "#92397a", "AR6-SSP5-8.5": "#980002",
}
RCP_PALETTE = "AR5"      # "AR5" (original RCP colours) or "AR6" (AR6 report's RCP colours)
SCENARIO_TO_IPCC = {
    "rcp26": f"{RCP_PALETTE}-RCP-2.6", "rcp45": f"{RCP_PALETTE}-RCP-4.5",
    "rcp60": f"{RCP_PALETTE}-RCP-6.0", "rcp85": f"{RCP_PALETTE}-RCP-8.5",
    "ssp119": "AR6-SSP1-1.9", "ssp126": "AR6-SSP1-2.6", "ssp245": "AR6-SSP2-4.5",
    "ssp370": "AR6-SSP3-7.0", "ssp434": "AR6-SSP4-3.4", "ssp460": "AR6-SSP4-6.0",
    "ssp534os": "AR6-SSP5-3.4-OS", "ssp585": "AR6-SSP5-8.5",
}
HIST_COLOR = "black"     # historical part of the curves (CRU or CMIP historical)
LINESTYLES = ["-", "--", "-.", ":"]


def model_list():
    models = []
    for e in EXPERIMENTS:
        if e.gcm not in models:
            models.append(e.gcm)
    return models


def experiment_color(exp):
    """IPCC colour of the scenario."""
    if exp.color:
        return exp.color
    key = SCENARIO_TO_IPCC.get(exp.scenario.lower())
    return IPCC_COLORS[key] if key else plt.get_cmap("tab10")(EXPERIMENTS.index(exp) % 10)


def experiment_linestyle(exp):
    """One line style per GCM (so several GCMs with the same scenario stay readable)."""
    return LINESTYLES[model_list().index(exp.gcm) % len(LINESTYLES)]


def experiment_label_base(exp):
    label = exp.label or SCENARIO_LABELS.get(exp.scenario.lower(), exp.scenario)
    return f"{exp.gcm} {label}" if len(model_list()) > 1 else label


def experiment_label(exp):
    label = experiment_label_base(exp)
    same = [e for e in EXPERIMENTS if experiment_label_base(e) == label]
    return label if len(same) == 1 else exp.name       # several members: use the name


In [ ]:
# ======================================================================
# 5. HELPERS: DOWNLOADS AND CMIP SOURCE RESOLUTION
# ======================================================================
class DownloadError(RuntimeError):
    """Download problem that retrying will not fix (404, login page, ...)."""


def is_url(value):
    return str(value).strip().lower().startswith(("http://", "https://"))


def download_file(url, target, timeout=60, retries=3, chunk_size=1 << 20):
    """Stream `url` to `target` (via a .part file, so no half-written files)."""
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    part = target.with_name(target.name + ".part")

    print(f"Downloading {url}")
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout) as response:
                if 400 <= response.status_code < 500:
                    raise DownloadError(
                        f"HTTP {response.status_code} for {url}. "
                        "If the server needs a login (ESGF, NASA Earthdata), "
                        "download the file in your browser and use the local path."
                    )
                response.raise_for_status()

                if "text/html" in response.headers.get("Content-Type", "").lower():
                    raise DownloadError(
                        f"{url} returned a web page instead of a file "
                        "(login or redirect page?). Download it manually "
                        "and use the local path."
                    )

                expected = response.headers.get("Content-Length")
                compressed = "Content-Encoding" in response.headers

                with open(part, "wb") as handle:
                    for block in response.iter_content(chunk_size):
                        handle.write(block)

            size = part.stat().st_size
            if expected and not compressed and size != int(expected):
                raise IOError(f"truncated download ({size} of {expected} bytes)")

            part.replace(target)
            print(f"  saved {target} ({size / 1e6:.1f} MB)")
            return target

        except DownloadError:
            part.unlink(missing_ok=True)
            raise
        except (requests.RequestException, IOError) as exc:
            last_error = exc
            part.unlink(missing_ok=True)
            print(f"  attempt {attempt}/{retries} failed: {exc}")
            time.sleep(2 * attempt)

    raise DownloadError(f"Could not download {url}: {last_error}")


def _looks_like_netcdf(path):
    with open(path, "rb") as handle:
        head = handle.read(8)
    return head[:3] == b"CDF" or head[:4] == b"\x89HDF"


def resolve_cmip_sources(source, data_dir=CMIP_DATA_DIR):
    """Turn a link / path / wildcard / list of those into a list of openable
    local paths (or OPeNDAP URLs). Links are downloaded once and cached."""
    if source is None:
        return []

    items = [source] if isinstance(source, (str, Path)) else list(source)
    resolved = []

    for item in items:
        text = str(item).strip()

        if is_url(text):
            if "/dodsc/" in text.lower():           # OPeNDAP: stream, don't download
                resolved.append(text)
                continue
            name = Path(urlparse(text).path).name or "cmip_download.nc"
            tag = hashlib.md5(text.encode()).hexdigest()[:8]
            target = Path(data_dir) / "downloads" / f"{tag}_{name}"
            if not (target.exists() and target.stat().st_size > 0):
                download_file(text, target)
                if not _looks_like_netcdf(target):      # e.g. a login page saved as .nc
                    target.unlink()
                    raise DownloadError(
                        f"{text} did not return a NetCDF file (login or error page?). "
                        "Download it manually and use the local path.")
            resolved.append(target)
            continue

        path = Path(text).expanduser()
        if not path.is_absolute() and not path.exists():
            path = Path(data_dir) / path            # relative to CMIP_DATA_DIR

        if any(ch in path.name for ch in "*?["):
            matches = sorted(path.parent.glob(path.name))
            if not matches:
                raise FileNotFoundError(f"No files match: {path}")
            resolved.extend(matches)
        elif path.exists():
            resolved.append(path)
        else:
            raise FileNotFoundError(f"CMIP file not found: {path}")

    return resolved


In [ ]:
# ======================================================================
# 6. CMIP INPUT ENGINE (no need to edit)
# ======================================================================
#
# Each CMIP file is opened ONCE per experiment, and every distinct grid cell is
# read, merged, unit-converted and validated ONCE (many glaciers share a cell).

def _open_cmip_dataset(path):
    path = str(path)
    try:
        return xr.open_dataset(
            path, decode_times=xr.coders.CFDatetimeCoder(use_cftime=True))
    except AttributeError:                                # older xarray
        return xr.open_dataset(path, decode_times=True, use_cftime=True)


def _wrap_dlon(lon_values, lon0):
    """Signed longitude difference in degrees, in [-180, 180)."""
    return (np.asarray(lon_values, dtype=float) - lon0 + 180.0) % 360.0 - 180.0


def _nearest_regular(lat_values, lon_values, lat0, lon0):
    """Nearest cell of a regular grid (separate 1-D lat and lon axes)."""
    i = int(np.argmin(np.abs(np.asarray(lat_values, dtype=float) - lat0)))
    j = int(np.argmin(np.abs(_wrap_dlon(lon_values, lon0))))
    return i, j


def _nearest_curvilinear(lat_values, lon_values, lat0, lon0):
    """Nearest cell on 2-D (or flattened 1-D) lat/lon arrays of equal shape."""
    lat_values = np.asarray(lat_values, dtype=float)
    dlat = lat_values - lat0
    dlon = _wrap_dlon(lon_values, lon0) * np.cos(np.deg2rad(lat0))
    dist = dlat ** 2 + dlon ** 2
    return tuple(int(i) for i in np.unravel_index(np.nanargmin(dist), dist.shape))


def _days_in_month(year, month, calendar):
    year = np.asarray(year)
    month = np.asarray(month)
    cal = (calendar or "standard").lower()
    days = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])[month - 1]

    if cal == "360_day":
        return np.full(days.shape, 30)
    if cal in ("noleap", "365_day"):
        leap = np.zeros(days.shape, dtype=bool)
    elif cal in ("all_leap", "366_day"):
        leap = np.ones(days.shape, dtype=bool)
    elif cal == "julian":
        leap = year % 4 == 0
    else:                                                  # (proleptic) gregorian
        leap = (year % 4 == 0) & ((year % 100 != 0) | (year % 400 == 0))
    return np.where((month == 2) & leap, 29, days)


def _temperature_to_kelvin(values, units):
    u = re.sub(r"[ _*^]", "", (units or "").lower())
    if u in ("k", "kelvin"):
        return values
    if u in ("c", "degc", "celsius", "degreec", "degreesc", "degreescelsius", "degreecelsius"):
        return values + 273.15
    raise ValueError(
        f"Unknown temperature units '{units}'. Set tas_units='K' or 'degC' "
        "in the CMIPExperiment."
    )


def _precip_to_mm_month(values, units, year, month, calendar):
    u = re.sub(r"[ *^()]", "", (units or "").lower())
    days = _days_in_month(year, month, calendar)

    if "month" in u:                                       # already per month
        return values
    if "day" in u or u.endswith("d-1") or u.endswith("/d"):
        return values * days
    if "s-1" in u or u.endswith("/s"):                     # kg m-2 s-1 or mm/s
        return values * days * 86400.0
    raise ValueError(
        f"Unknown precipitation units '{units}'. Set pr_units "
        "(e.g. 'kg m-2 s-1' or 'mm/day') in the CMIPExperiment."
    )


def _trim_part(part, min_key=None, max_key=None):
    """Keep only months with min_key <= year*12+month-1 <= max_key.
    Returns None when nothing is left."""
    key = part["year"] * 12 + (part["month"] - 1)
    keep = np.ones(key.shape, dtype=bool)
    if min_key is not None:
        keep &= key >= min_key
    if max_key is not None:
        keep &= key <= max_key
    if not keep.any():
        return None
    out = dict(part)
    for k in ("time", "year", "month", "values"):
        out[k] = part[k][keep]
    return out


def _merge_monthly(parts, what):
    """Merge monthly series (historical + scenario, or time-chunked files).
    Sorted by month, duplicate months keep the FIRST part, trimmed to complete
    calendar years, and checked for gaps."""
    time_obj = np.concatenate([p["time"] for p in parts])
    year = np.concatenate([p["year"] for p in parts]).astype(int)
    month = np.concatenate([p["month"] for p in parts]).astype(int)
    values = np.concatenate([p["values"] for p in parts])

    key = year * 12 + (month - 1)
    order = np.argsort(key, kind="stable")
    _, first = np.unique(key[order], return_index=True)
    sel = order[first]

    complete = [y for y, n in zip(*np.unique(year[sel], return_counts=True)) if n == 12]
    sel = sel[np.isin(year[sel], complete)]
    if sel.size == 0:
        raise ValueError(f"{what}: no complete calendar year found.")

    key = key[sel]
    gaps = np.where(np.diff(key) != 1)[0]
    if gaps.size:
        k = int(key[gaps[0]])
        raise ValueError(
            f"{what}: months are missing after {k // 12}-{k % 12 + 1:02d}. "
            "Check that the historical and scenario files join up."
        )

    return {
        "time": time_obj[sel], "year": year[sel], "month": month[sel],
        "key": key, "values": values[sel],
        "units": parts[0]["units"], "lat": parts[0]["lat"], "lon": parts[0]["lon"],
        "calendar": parts[0]["calendar"], "time_unit": parts[0]["time_unit"],
    }


class CMIPForcing:
    """Climate forcing of one CMIPExperiment, shared by all glaciers."""

    _KEYS = ("historical_temp", "historical_prcp", "future_temp", "future_prcp")

    def __init__(self, experiment, data_dir=CMIP_DATA_DIR, end_year=FUTURE_END_YEAR,
                 start_year=None):
        self.exp = experiment
        self.start_year = start_year          # first year the forcing must cover (None: bias period)
        self.end_year = experiment.end_year or end_year

        self.files = {k: resolve_cmip_sources(getattr(experiment, k), data_dir)
                      for k in self._KEYS}
        for k in ("future_temp", "future_prcp"):
            if not self.files[k]:
                raise ValueError(f"{experiment.name}: '{k}' is required.")

        self._ds = {k: [_open_cmip_dataset(p) for p in v] for k, v in self.files.items()}
        self._check_grids()
        self._cells = {}

    def close(self):
        for datasets in self._ds.values():
            for ds in datasets:
                ds.close()

    def _check_grids(self):
        exp = self.exp
        ref = self._ds["future_temp"][0]
        for k, datasets in self._ds.items():
            for ds in datasets:
                for coord in (exp.lat_name, exp.lon_name):
                    if coord not in ds.coords and coord not in ds:
                        raise KeyError(
                            f"{exp.name}: coordinate '{coord}' not found. "
                            f"Available: {list(ds.coords)}. Set lat_name/lon_name.")
                    a, b = ref[coord].values, ds[coord].values
                    if a.shape != b.shape or not np.allclose(a, b, equal_nan=True):
                        raise ValueError(
                            f"{exp.name}: '{k}' is on a different grid than the "
                            "scenario temperature file. All files of one "
                            "experiment must share one grid.")

    def _nearest_indexers(self, lat, lon):
        exp = self.exp
        ds = self._ds["future_temp"][0]
        lat_c, lon_c = ds[exp.lat_name], ds[exp.lon_name]

        if lat_c.ndim == 1 and lon_c.ndim == 1 and lat_c.dims != lon_c.dims:
            i, j = _nearest_regular(lat_c.values, lon_c.values, lat, lon)
            return {lat_c.dims[0]: i, lon_c.dims[0]: j}

        if lat_c.dims != lon_c.dims:
            raise ValueError(
                f"Unsupported grid: lat dims {lat_c.dims}, lon dims {lon_c.dims}")
        idx = _nearest_curvilinear(lat_c.values, lon_c.values, lat, lon)
        return dict(zip(lat_c.dims, idx))

    def _read_part(self, ds, variable, indexers):
        exp = self.exp
        if variable not in ds:
            raise KeyError(f"Variable '{variable}' not found. "
                           f"Available: {list(ds.data_vars)}")

        def pick(da):
            return da.isel({d: i for d, i in indexers.items() if d in da.dims})

        da = pick(ds[variable])
        for dim in [d for d in da.dims if d != exp.time_name]:
            if da.sizes[dim] == 1:
                da = da.isel({dim: 0})
        if da.dims != (exp.time_name,):
            raise ValueError(f"Expected a (time,) series for '{variable}', got {da.dims}")
        da = da.load()

        t = da[exp.time_name]
        times = t.values
        return {
            "time": times,
            "year": t.dt.year.values.astype(int),
            "month": t.dt.month.values.astype(int),
            "values": np.asarray(da.values, dtype=float),
            "units": da.attrs.get("units", ""),
            "lat": float(pick(ds[exp.lat_name]).values),
            "lon": float(pick(ds[exp.lon_name]).values),
            "calendar": getattr(times[0], "calendar", None) or t.encoding.get("calendar") or "standard",
            "time_unit": t.encoding.get("units"),
        }

    def _build_cell(self, indexers):
        exp = self.exp
        merged = {}
        for kind, var in (("temp", exp.tas_var), ("prcp", exp.pr_var)):
            hist_ds = self._ds["historical_" + kind]
            fut_ds = self._ds["future_" + kind]
            h_end = exp.historical_end[0] * 12 + exp.historical_end[1] - 1
            f_start = exp.scenario_start[0] * 12 + exp.scenario_start[1] - 1

            # historical files: up to Dec 2014; scenario files: from Jan 2015
            # (only when historical files are given; otherwise use the
            # scenario file as it is, e.g. cmip5-ng files cover both)
            parts = [_trim_part(self._read_part(ds, var, indexers), max_key=h_end)
                     for ds in hist_ds]
            parts += [_trim_part(self._read_part(ds, var, indexers),
                                 min_key=f_start if hist_ds else None)
                      for ds in fut_ds]
            parts = [p for p in parts if p is not None]
            if hist_ds and len(parts) < 2:
                raise ValueError(
                    f"{exp.name} {kind}: no data left after cutting the historical "
                    f"files at {exp.historical_end} and the scenario files from "
                    f"{exp.scenario_start}. Check the files / dates.")
            merged[kind] = _merge_monthly(parts, f"{exp.name} {kind}")

        mt, mp = merged["temp"], merged["prcp"]
        if not np.array_equal(mt["key"], mp["key"]):
            raise ValueError(f"{exp.name}: temperature and precipitation "
                             "cover different months.")

        calendar = exp.calendar or mt["calendar"]
        temp = _temperature_to_kelvin(mt["values"], exp.tas_units or mt["units"])
        prcp = _precip_to_mm_month(mp["values"], exp.pr_units or mp["units"],
                                   mp["year"], mp["month"], calendar)

        if not (np.isfinite(temp).all() and np.isfinite(prcp).all()):
            raise ValueError(f"{exp.name}: non-finite values (masked grid cell?).")
        if prcp.min() < 0:
            if prcp.min() < -1e-3:
                raise ValueError(f"{exp.name}: negative precipitation ({prcp.min():.3g} mm/month).")
            prcp = np.clip(prcp, 0, None)       # tiny numerical negatives

        first_year, last_year = int(mt["year"][0]), int(mt["year"][-1])
        needed = int(exp.bias_correction_period[0])
        if self.start_year is not None:
            needed = min(needed, int(self.start_year))
        if first_year > needed:
            raise ValueError(f"{exp.name}: data start in {first_year}, but the "
                             f"forcing is needed from {needed} (bias-correction period "
                             f"{exp.bias_correction_period}, model start). "
                             "Add the historical files.")
        if last_year < self.end_year:
            raise ValueError(f"{exp.name}: data end in {last_year}, before "
                             f"FUTURE_END_YEAR={self.end_year}. Lower end_year "
                             "for this experiment.")

        coords = {"time": mt["time"], "lat": mt["lat"], "lon": mt["lon"]}
        return {
            "temp": xr.DataArray(temp, dims=("time",), coords=coords, name="tas",
                                 attrs={"units": "K"}),
            "prcp": xr.DataArray(prcp, dims=("time",), coords=coords, name="pr",
                                 attrs={"units": "mm month-1"}),
            "calendar": calendar,
            "time_unit": exp.time_unit or mt["time_unit"],
        }

    def get_forcing(self, lat, lon):
        indexers = self._nearest_indexers(lat, lon)
        key = tuple(sorted(indexers.items()))
        if key not in self._cells:
            try:
                self._cells[key] = self._build_cell(indexers)
            except Exception as exc:                    # cache failures too
                self._cells[key] = exc
        cell = self._cells[key]
        if isinstance(cell, Exception):
            raise cell
        return cell


def _process_gcm_signature():
    """OGGM versions differ (filesuffix vs output_filesuffix, optional
    apply_bias_correction / time_unit): look at what is really supported."""
    return inspect.signature(gcm_climate.process_gcm_data).parameters


def add_cmip_forcing(gdir, forcing):
    """Bias-correct the GCM series of this glacier's grid cell against CRU and
    write it as gcm_data<suffix>."""
    exp = forcing.exp
    cell = forcing.get_forcing(float(gdir.cenlat), float(gdir.cenlon))
    params = _process_gcm_signature()

    kwargs = {
        "prcp": cell["prcp"],
        "temp": cell["temp"],
        "year_range": exp.bias_correction_period,
        "scale_stddev": exp.scale_stddev,
        "source": exp.source or exp.name,
        ("output_filesuffix" if "output_filesuffix" in params else "filesuffix"): exp.suffix,
    }
    if "calendar" in params and cell["calendar"]:
        kwargs["calendar"] = cell["calendar"]
    if "time_unit" in params and cell["time_unit"]:
        kwargs["time_unit"] = cell["time_unit"]
    if "apply_bias_correction" in params:
        kwargs["apply_bias_correction"] = exp.apply_bias_correction
    elif not exp.apply_bias_correction:
        raise NotImplementedError(
            "This OGGM version has no apply_bias_correction option; "
            "set it to True or use OGGM >= 1.6.")

    gcm_climate.process_gcm_data(gdir, **kwargs)
    return gdir.get_filepath("gcm_data", filesuffix=exp.suffix)


In [ ]:
# ======================================================================
# 7. INITIALIZE OGGM-VAS
# ======================================================================
print("\nInitializing OGGM-VAS...")

vascaling.initialize()

cfg.PATHS["working_dir"] = str(WORKDIR)

cfg.PARAMS["rgi_version"] = RGI_VERSION
cfg.PARAMS["baseline_climate"] = "CRU"      # reference climate (calibration + bias correction)
cfg.PARAMS["hydro_month_nh"] = 1            # calendar years
cfg.PARAMS["hydro_month_sh"] = 1
cfg.PARAMS["use_multiprocessing"] = False
cfg.PARAMS["use_intersects"] = False
cfg.PARAMS["continue_on_error"] = True      # a failing glacier does not stop the run

# VAS parameters
cfg.PARAMS["prcp_scaling_factor"] = 3.0
cfg.PARAMS["temp_melt"] = 0.0
cfg.PARAMS["temp_all_solid"] = 4.0
cfg.PARAMS["temp_all_liq"] = 2.0
cfg.PARAMS["climate_qc_months"] = 3
cfg.PARAMS["temp_default_gradient"] = -0.0065
cfg.PARAMS["temp_local_gradient_bounds"] = [-0.009, -0.003]
cfg.PARAMS["temp_use_local_gradient"] = False
cfg.PARAMS["prcp_default_gradient"] = 0.0004
cfg.PARAMS["mu_star_halfperiod"] = 15
cfg.PARAMS["min_mu_star"] = 5
cfg.PARAMS["max_mu_star"] = 10000
cfg.PARAMS["use_bias_for_run"] = True       # use the calibrated VAS bias in model runs
cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["mp_processes"] = 4

In [ ]:
# ======================================================================
# 8. LOAD RGI 7.0 REGION
# ======================================================================
print(f"\nLoading RGI 7.0 region {RGI_REGION:02d}: {REGION_NAME}")

# OGGM 1.6.3 downloads the official RGI7 archive and extracts the region
rgi_file = utils.get_rgi_region_file(RGI_REGION, version=RGI_VERSION)
print("RGI file:", rgi_file)

rgidf = gpd.read_file(rgi_file)
print("Number of RGI7 glaciers:", len(rgidf))

# glacier id
if "rgi_id" not in rgidf.columns:
    if "RGIId" in rgidf.columns:
        rgidf["rgi_id"] = rgidf["RGIId"]
    else:
        raise RuntimeError("Could not identify the RGI7 glacier ID column.")

# RGI7 has no nominal glaciers; only filter when an RGI6-style status field exists
status_column = next((c for c in ("status", "Status") if c in rgidf.columns), None)
if status_column is not None:
    nominal = rgidf[status_column].astype(str).str.strip().isin(["2", "9"])
    print("Nominal/unassigned glaciers removed:", int(nominal.sum()))
    rgidf = rgidf.loc[~nominal].copy()

if MAX_GLACIERS is not None:
    rgidf = rgidf.head(MAX_GLACIERS).copy()
    print("TEST MODE: using", len(rgidf), "glaciers")
print("Glaciers to model:", len(rgidf))


In [ ]:
# ======================================================================
# 9. GLACIER DIRECTORIES, MASKS, CRU CLIMATE
# ======================================================================
#
# RGI-TOPO provides the official OGGM level-1 glacier directories for RGI7.

PREPRO_BASE_URL = ("https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/"
                   "rgitopo/2026.1/all_dems/")

gdirs = workflow.init_glacier_directories(
    rgidf,
    from_prepro_level=1,
    prepro_border=10,
    prepro_rgi_version=RGI_VERSION,
    prepro_base_url=PREPRO_BASE_URL,
)
print("Glacier directories:", len(gdirs))


def run_missing(task, gdirs, filename, **kwargs):
    """Run an entity task only for glaciers whose output file is missing."""
    if RECOMPUTE_BASE:
        todo = list(gdirs)
    else:
        todo = [g for g in gdirs if not os.path.exists(g.get_filepath(filename))]
    print(f"{task.__name__}: {len(gdirs) - len(todo)} done already, {len(todo)} to run")
    if todo:
        workflow.execute_entity_task(task, todo, **kwargs)


run_missing(tasks.glacier_masks, gdirs, "gridded_data")

# gridded_data.nc is required later: rebuild it where it is still missing
broken = [g for g in gdirs if not os.path.exists(g.get_filepath("gridded_data"))]
if broken:
    print(f"Rebuilding gridded_data for {len(broken)} glaciers...")
    for g in broken:
        try:
            tasks.glacier_masks(g, reset=True)
        except Exception as exc:
            print(f"  {g.rgi_id}: {exc}")

# CRU is the reference climate for calibration and bias correction (and, in
# CRU mode, the forcing of the reconstruction)
run_missing(tasks.process_climate_data, gdirs, "climate_historical",
            y0=MODEL_START_YEAR, y1=CRU_RECON_END_YEAR)


In [ ]:
# ======================================================================
# 10. VAS CALIBRATION (RGI6/CRU reference t* table applied spatially to RGI7)
# ======================================================================
ref_table_file = vascaling.get_ref_tstars_filepath(VAS_REFERENCE_TSTARS_FILE)
ref_df = pd.read_csv(ref_table_file)
print("VAS reference table:", ref_table_file, f"({len(ref_df)} reference glaciers)")

missing_columns = {"lon", "lat", "tstar", "bias"} - set(ref_df.columns)
if missing_columns:
    raise RuntimeError(f"The VAS reference table is missing columns: {missing_columns}")
ref_df.to_csv(WORKDIR / "vas_ref_tstars_rgi7_spatial_cru.csv", index=False)

cfg.PARAMS["run_mb_calibration"] = False
run_missing(vascaling.local_t_star, gdirs, "vascaling_mustar", ref_df=ref_df)

# keep only successfully calibrated glaciers
records, valid_gdirs, failed_calibration = [], [], []
for gdir in gdirs:
    try:
        cal = gdir.read_json("vascaling_mustar")
    except Exception:
        failed_calibration.append(gdir.rgi_id)
        continue
    rec = {"rgi_id": gdir.rgi_id,
           "t_star": cal.get("t_star", np.nan),
           "bias": cal.get("bias", np.nan),
           "mu_star": cal.get("mu_star", np.nan)}
    records.append(rec)
    if np.isfinite([rec["t_star"], rec["bias"], rec["mu_star"]]).all():
        valid_gdirs.append(gdir)
    else:
        failed_calibration.append(gdir.rgi_id)

pd.DataFrame(records).to_csv(WORKDIR / "vas_calibration_rgi70.csv", index=False)
pd.DataFrame({"rgi_id": failed_calibration}).to_csv(WORKDIR / "failed_vas_calibration.csv", index=False)
print(f"VAS calibration: {len(valid_gdirs)} valid, {len(failed_calibration)} failed")

gdirs = valid_gdirs
if not gdirs:
    raise RuntimeError("No glaciers passed the VAS calibration.")


In [ ]:
# ======================================================================
# 11. RESULT HELPERS
# ======================================================================
# column name -> (model_diagnostics variable, unit factor)
DIAG_VARS = {
    "volume_km3": ("volume_m3", 1e-9),
    "area_km2": ("area_m2", 1e-6),
    "length_km": ("length_m", 1e-3),
    "specific_mb_mmwe": ("spec_mb", 1.0),
}


def read_diagnostics(gdir, suffix):
    with xr.open_dataset(gdir.get_filepath("model_diagnostics", filesuffix=suffix)) as ds:
        data = {"year": np.asarray(ds["time"].values, dtype=float).astype(int)}
        for column, (variable, factor) in DIAG_VARS.items():
            if variable in ds:
                data[column] = ds[variable].values * factor
    df = pd.DataFrame(data)
    df["rgi_id"] = gdir.rgi_id
    return df


def collect_results(gdirs, suffix):
    """Read the yearly diagnostics of all glaciers that have them."""
    frames, failed = [], []
    for i, gdir in enumerate(gdirs, start=1):
        try:
            frames.append(read_diagnostics(gdir, suffix))
        except Exception as exc:
            failed.append((gdir.rgi_id, str(exc)))
        if i % 500 == 0:
            print(f"  read {i}/{len(gdirs)} glaciers")
    if not frames:
        return pd.DataFrame(), failed
    df = pd.concat(frames, ignore_index=True).sort_values(["rgi_id", "year"]).reset_index(drop=True)
    return df, failed


def aggregate_region(glacier_df):
    """Regional sums. Length is the SUM of glacier lengths (total length)."""
    return (glacier_df.groupby("year", as_index=False)
            .agg(volume_km3=("volume_km3", "sum"),
                 area_km2=("area_km2", "sum"),
                 length_km=("length_km", "sum"))
            .sort_values("year").reset_index(drop=True))


def add_mass_and_sle(df, volume_ref_km3):
    """Mass loss and sea-level equivalent relative to a reference volume."""
    df = df.copy()
    loss = volume_ref_km3 - df["volume_km3"]
    df["volume_loss_km3"] = loss
    df["mass_loss_Gt"] = loss * RHO_ICE / RHO_WATER          # 1 km3 water = 1 Gt
    df["sea_level_change_mm"] = df["mass_loss_Gt"] / OCEAN_AREA_KM2 * 1e6
    df["annual_mass_change_Gt_yr"] = df["mass_loss_Gt"].diff()   # positive = loss
    df["annual_sle_mm_yr"] = df["sea_level_change_mm"].diff()
    return df


def volume_at(df, year):
    return float(df.loc[df["year"] == year, "volume_km3"].iloc[0])


def make_regional_hist(glacier_df):
    """Regional historical series with changes relative to MODEL_START_YEAR."""
    regional = aggregate_region(glacier_df)
    start = regional.loc[regional["year"] == MODEL_START_YEAR]
    if start.empty:
        raise RuntimeError(f"No results for {MODEL_START_YEAR}.")
    start = start.iloc[0]
    regional["volume_change_km3"] = regional["volume_km3"] - start["volume_km3"]
    regional["area_change_km2"] = regional["area_km2"] - start["area_km2"]
    regional["length_change_km"] = regional["length_km"] - start["length_km"]
    return add_mass_and_sle(regional, float(start["volume_km3"]))


def summarize_errors(errors, limit=3):
    """errors: list of (rgi_id, message). Print the most common messages."""
    if not errors:
        return
    counts = pd.Series([m for _, m in errors]).value_counts()
    for message, n in counts.head(limit).items():
        print(f"  {n} x {message[:200]}")


In [ ]:
# ======================================================================
# 12. CRU RECONSTRUCTION (only when HISTORICAL_FORCING = "CRU")
# ======================================================================
reconstruction = None          # regional CRU reconstruction (CRU mode)
reconstruction_glacier = None  # glacier-level CRU reconstruction (CRU mode)

if not CMIP_MODE:
    print(f"\nVAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (CRU)")

    if RECOMPUTE_BASE:
        todo = list(gdirs)
    else:
        todo = [g for g in gdirs if not os.path.exists(
            g.get_filepath("model_diagnostics", filesuffix=CRU_HIST_SUFFIX))]
    print(f"{len(gdirs) - len(todo)} done already, {len(todo)} to run")

    if todo:
        workflow.execute_entity_task(
            vascaling.run_historic_from_climate_data,
            todo,
            ys=MODEL_START_YEAR,
            ye=CRU_RECON_END_YEAR,
            climate_filename="climate_historical",
            output_filesuffix=CRU_HIST_SUFFIX,
        )

    reconstruction_glacier, failed_hist = collect_results(gdirs, CRU_HIST_SUFFIX)
    if reconstruction_glacier.empty:
        raise RuntimeError("No successful historical VAS runs.")

    ok_ids = set(reconstruction_glacier["rgi_id"])
    pd.DataFrame({"rgi_id": [g.rgi_id for g in gdirs if g.rgi_id not in ok_ids]}).to_csv(
        WORKDIR / "failed_historical_rgi70.csv", index=False)
    gdirs = [g for g in gdirs if g.rgi_id in ok_ids]
    print(f"Historical runs: {len(gdirs)} successful, {len(failed_hist)} failed")

    reconstruction = make_regional_hist(reconstruction_glacier)
    reconstruction_glacier.to_csv(
        WORKDIR / f"glacier_results_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}.csv", index=False)
    reconstruction.to_csv(
        WORKDIR / f"regional_results_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}.csv", index=False)

    last = reconstruction.loc[reconstruction["year"] == CRU_RECON_END_YEAR].iloc[0]
    print(f"Volume {MODEL_START_YEAR}: {volume_at(reconstruction, MODEL_START_YEAR):.3f} km3 | "
          f"{CRU_RECON_END_YEAR}: {last['volume_km3']:.3f} km3 | "
          f"SLE: {last['sea_level_change_mm']:.5f} mm")


In [ ]:
# ======================================================================
# 13. RUN ALL EXPERIMENTS
# ======================================================================
#
# per experiment: GCM bias correction -> [CMIP-forced historical run] -> projection

results = {}        # experiment name -> {"hist": DataFrame, "future": DataFrame}
failures = {}       # experiment name -> reason


def run_experiment(exp, gdirs):
    end_year = exp.end_year or FUTURE_END_YEAR
    hist_csv = WORKDIR / f"regional_results_hist_{exp.name}.csv"
    future_csv = WORKDIR / f"regional_results_future_{exp.name}.csv"

    # ---- reuse a finished experiment --------------------------------------
    if REUSE_RESULTS and future_csv.exists() and (not CMIP_MODE or hist_csv.exists()):
        future = pd.read_csv(future_csv)
        hist = pd.read_csv(hist_csv) if CMIP_MODE else reconstruction
        fy, hy = set(future["year"].astype(int)), set(hist["year"].astype(int))
        if {SPLIT_YEAR, end_year} <= fy and {MODEL_START_YEAR, SPLIT_YEAR} <= hy:
            print("Existing results found, reusing:", future_csv.name)
            ref = volume_at(hist, MODEL_START_YEAR)
            return {"hist": add_mass_and_sle(hist, ref) if CMIP_MODE else hist,
                    "future": add_mass_and_sle(future, ref)}

    # ---- 1. GCM forcing, bias-corrected against CRU ------------------------
    def has_forcing(g):
        return os.path.exists(g.get_filepath("gcm_data", filesuffix=exp.suffix))

    missing = list(gdirs) if OVERWRITE_CMIP else [g for g in gdirs if not has_forcing(g)]
    print(f"GCM forcing: {len(gdirs) - len(missing)} done already, {len(missing)} to process")

    forcing_errors = []
    if missing:
        forcing = CMIPForcing(exp, end_year=end_year,
                              start_year=MODEL_START_YEAR if CMIP_MODE else None)
        try:
            for i, gdir in enumerate(missing, start=1):
                try:
                    add_cmip_forcing(gdir, forcing)
                except Exception as exc:
                    forcing_errors.append((gdir.rgi_id, f"{type(exc).__name__}: {exc}"))
                if i % 250 == 0:
                    print(f"  forcing {i}/{len(missing)}")
        finally:
            forcing.close()

    if forcing_errors:
        print(f"GCM forcing failed for {len(forcing_errors)} glaciers:")
        summarize_errors(forcing_errors)
        pd.DataFrame(forcing_errors, columns=["rgi_id", "error"]).to_csv(
            WORKDIR / f"failed_cmip_{exp.name}.csv", index=False)

    forced = [g for g in gdirs if has_forcing(g)]
    if not forced:
        raise RuntimeError("no glacier has GCM forcing")

    # ---- 2. historical run -------------------------------------------------
    if CMIP_MODE:
        def has_hist(g):
            return os.path.exists(g.get_filepath("model_diagnostics", filesuffix=exp.hist_suffix))

        redo = OVERWRITE_CMIP or RECOMPUTE_BASE
        todo = forced if redo else [g for g in forced if not has_hist(g)]
        print(f"CMIP-forced VAS run {MODEL_START_YEAR}-{SPLIT_YEAR}: "
              f"{len(forced) - len(todo)} done already, {len(todo)} to run")
        if todo:
            workflow.execute_entity_task(
                vascaling.run_historic_from_climate_data,
                todo,
                ys=MODEL_START_YEAR,
                ye=SPLIT_YEAR,
                climate_filename="gcm_data",
                climate_input_filesuffix=exp.suffix,
                output_filesuffix=exp.hist_suffix,
            )

        hist_glacier, _ = collect_results(forced, exp.hist_suffix)
        if hist_glacier.empty:
            raise RuntimeError("no successful CMIP-forced historical runs")
        hist_regional = make_regional_hist(hist_glacier)
        hist_glacier.to_csv(WORKDIR / f"glacier_results_hist_{exp.name}.csv", index=False)
        hist_regional.to_csv(hist_csv, index=False)

        ok_ids = set(hist_glacier["rgi_id"])
        base = [g for g in forced if g.rgi_id in ok_ids]
        init_suffix = exp.hist_suffix
    else:
        hist_glacier, hist_regional = reconstruction_glacier, reconstruction
        base = forced
        init_suffix = CRU_HIST_SUFFIX

    volume_start = volume_at(hist_regional, MODEL_START_YEAR)

    # ---- 3. projection from the end of the historical run -------------------
    print(f"VAS projection {SPLIT_YEAR}-{end_year} for {len(base)} glaciers")
    models = workflow.execute_entity_task(
        vascaling.run_from_climate_data,
        base,
        ys=SPLIT_YEAR,
        ye=end_year,
        climate_filename="gcm_data",
        climate_input_filesuffix=exp.suffix,
        output_filesuffix=exp.future_suffix,
        init_model_filesuffix=init_suffix,
        init_model_yr=SPLIT_YEAR,
    )
    ran = [g for g, m in zip(base, models) if m is not None]

    future_glacier, _ = collect_results(ran, exp.future_suffix)
    if future_glacier.empty:
        raise RuntimeError("no successful projection runs")

    # ---- 4. glaciers lost between historical run and projection ---------------
    start_vol = hist_glacier.loc[hist_glacier["year"] == SPLIT_YEAR]
    lost = ~start_vol["rgi_id"].isin(set(future_glacier["rgi_id"]))
    lost_fraction = float(start_vol.loc[lost, "volume_km3"].sum() / start_vol["volume_km3"].sum())
    pd.DataFrame({"rgi_id": start_vol.loc[lost, "rgi_id"]}).to_csv(
        WORKDIR / f"failed_future_{exp.name}.csv", index=False)
    if lost.any():
        print(f"WARNING: {int(lost.sum())} glaciers ({lost_fraction:.1%} of the {SPLIT_YEAR} "
              "volume) are missing in this projection; the regional curve will drop "
              "at the transition year.")

    # ---- 5. regional aggregation, save -------------------------------------
    future = aggregate_region(future_glacier)
    future["lost_volume_fraction"] = lost_fraction
    future = add_mass_and_sle(future, volume_start)

    future_glacier.to_csv(WORKDIR / f"glacier_results_future_{exp.name}.csv", index=False)
    future.to_csv(future_csv, index=False)
    print("Saved:", future_csv.name)
    return {"hist": hist_regional, "future": future}


if not RUN_FUTURE:
    print("RUN_FUTURE is False: nothing to do.")
elif not EXPERIMENTS:
    print("EXPERIMENTS is empty: add entries in the CMIP DATA cell.")
else:
    for exp in EXPERIMENTS:
        print("\n" + "=" * 60)
        print(f"{exp.name}  ({exp.generation or 'CMIP'} {exp.gcm} {exp.scenario})")
        print("=" * 60)
        try:
            results[exp.name] = run_experiment(exp, gdirs)
        except Exception as exc:
            failures[exp.name] = f"{type(exc).__name__}: {exc}"
            print(f"EXPERIMENT FAILED: {failures[exp.name]}")

    print(f"\nFinished: {len(results)} ok, {len(failures)} failed")
    for name, reason in failures.items():
        print(f"  {name}: {reason}")


In [ ]:
# ======================================================================
# 14. REGIONAL CRU CLIMATE SERIES (area-weighted)
# ======================================================================
#
# CRU climate of the glacier grid cells (as stored in climate_historical),
# weighted by RGI glacier area.

ref_start, ref_end = int(BIAS_CORRECTION_PERIOD[0]), int(BIAS_CORRECTION_PERIOD[1])

frames = []
for gdir in gdirs:
    climate_file = gdir.get_filepath("climate_historical")
    if not os.path.exists(climate_file):
        continue
    try:
        with xr.open_dataset(climate_file) as ds:
            df = pd.DataFrame({"year": ds["time"].dt.year.values,
                               "temp": ds["temp"].values,
                               "prcp": ds["prcp"].values})
    except Exception as exc:
        print(f"Could not read CRU for {gdir.rgi_id}: {exc}")
        continue
    annual = df.groupby("year").agg(temp=("temp", "mean"), prcp=("prcp", "sum")).reset_index()
    annual["w"] = float(gdir.rgi_area_km2)
    frames.append(annual)

cru_all = pd.concat(frames, ignore_index=True)


def weighted_mean_by_year(df, column):
    d = df[np.isfinite(df[column]) & np.isfinite(df["w"]) & (df["w"] > 0)]
    return (d[column] * d["w"]).groupby(d["year"]).sum() / d["w"].groupby(d["year"]).sum()


cru_regional = pd.DataFrame({
    "cru_temperature_C": weighted_mean_by_year(cru_all, "temp"),
    "cru_precipitation_mm": weighted_mean_by_year(cru_all, "prcp"),
}).reset_index()

ref = cru_regional[cru_regional["year"].between(ref_start, ref_end)]
cru_regional["cru_temperature_anomaly_C"] = (
    cru_regional["cru_temperature_C"] - ref["cru_temperature_C"].mean())
cru_regional["cru_precipitation_anomaly_percent"] = (
    cru_regional["cru_precipitation_mm"] / ref["cru_precipitation_mm"].mean() - 1) * 100

cru_regional.to_csv(WORKDIR / f"regional_cru_climate_{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}.csv", index=False)
print("Regional CRU climate series saved.")


In [ ]:
# ======================================================================
# 15. PLOTS: RECONSTRUCTION + PROJECTIONS (official IPCC colours)
# ======================================================================
COMBINED_PLOTS = [
    # column, y label, file key, draw zero line
    ("volume_km3", "Glacier volume (km³)", "volume", False),
    ("area_km2", "Glacier area (km²)", "area", False),
    ("length_km", "Sum of glacier lengths (km)", "length", False),
    ("mass_loss_Gt", f"Cumulative glacier mass loss since {MODEL_START_YEAR} (Gt)", "mass_loss", True),
    ("sea_level_change_mm", f"Sea-level equivalent relative to {MODEL_START_YEAR} (mm)", "sea_level", True),
]


def plot_combined(column, ylabel, key, zero):
    fig, ax = plt.subplots(figsize=(11, 6))
    models = model_list()

    # historical part
    if not CMIP_MODE:
        ax.plot(reconstruction["year"], reconstruction[column], color=HIST_COLOR,
                linewidth=2, label=f"VAS reconstruction (CRU)")
    else:
        drawn = set()
        for exp in EXPERIMENTS:
            if exp.name in results and exp.gcm not in drawn:
                drawn.add(exp.gcm)
                hist = results[exp.name]["hist"]
                ax.plot(hist["year"], hist[column], color=HIST_COLOR,
                        linestyle=experiment_linestyle(exp), linewidth=2,
                        label=f"Historical {exp.gcm}" if len(models) > 1 else "Historical (CMIP)")

    # projections in IPCC colours
    for exp in EXPERIMENTS:
        if exp.name in results:
            fut = results[exp.name]["future"]
            ax.plot(fut["year"], fut[column], color=experiment_color(exp),
                    linestyle=experiment_linestyle(exp), linewidth=2,
                    label=experiment_label(exp))

    ax.axvline(SPLIT_YEAR, color="black", linestyle=":", linewidth=1)
    if zero:
        ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(f"RGI70-{RGI_REGION:02d} - {REGION_NAME}\n"
                 f"VAS reconstruction ({HISTORICAL_FORCING.upper()}-forced) and CMIP projections")
    ax.set_xlim(MODEL_START_YEAR, x_max)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(WORKDIR / f"{REGION_SLUG}_{key}_{MODEL_START_YEAR}_{x_max}_{HISTORICAL_FORCING.lower()}.png", dpi=300)
    plt.show()
    plt.close(fig)


if results:
    x_max = int(max(r["future"]["year"].max() for r in results.values()))
    for column, ylabel, key, zero in COMBINED_PLOTS:
        plot_combined(column, ylabel, key, zero)


In [ ]:
# ======================================================================
# 16. SUMMARY
# ======================================================================
rows = []
for exp in EXPERIMENTS:
    r = results.get(exp.name)
    if r is None:
        continue
    fut, hist = r["future"], r["hist"]
    first = fut.loc[fut["year"] == SPLIT_YEAR].iloc[0]
    last = fut.iloc[-1]
    rows.append({
        "RGI_region": RGI_REGION,
        "region_name": REGION_NAME,
        "experiment": exp.name,
        "gcm": exp.gcm,
        "generation": exp.generation,
        "scenario": exp.scenario,
        "historical_forcing": HISTORICAL_FORCING.upper(),
        f"volume_{MODEL_START_YEAR}_km3": volume_at(hist, MODEL_START_YEAR),
        "split_year": SPLIT_YEAR,
        "end_year": int(last["year"]),
        "volume_split_km3": first["volume_km3"],
        "volume_end_km3": last["volume_km3"],
        "area_split_km2": first["area_km2"],
        "area_end_km2": last["area_km2"],
        "sea_level_end_mm": last["sea_level_change_mm"],
        "lost_volume_fraction": fut["lost_volume_fraction"].iloc[0]
        if "lost_volume_fraction" in fut else np.nan,
    })

summary_df = pd.DataFrame(rows)
if not summary_df.empty:
    summary_df.to_csv(WORKDIR / "projection_summary.csv", index=False)
    print("\n" + "=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    print(summary_df.to_string(index=False))

print("\nRGI7 + VAS workflow complete.")
print(f"Region: {RGI_REGION:02d} - {REGION_NAME} | historical forcing: {HISTORICAL_FORCING.upper()}")
print(f"Historical: {MODEL_START_YEAR}-{SPLIT_YEAR} | projection: {SPLIT_YEAR}-{FUTURE_END_YEAR}")
print(f"Output directory: {WORKDIR}")
